In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

PATH = '/home/skrhakv/Projects/seq2pocket/data/'
AF2BIND_PREDICTIONS_PATH = f'{PATH}/predictions/af2bind/af2bind_p2rank_human_proteome_predictions_revision_with_sitemap.csv'
binding_pockets_df = pd.read_csv(AF2BIND_PREDICTIONS_PATH, sep=",", header=0)

# just 15k of unique uniprot accessions, not whole human proteome (that would be around 20k)
binding_pockets_df[['uniprot']].nunique()

/tmp/slurm.23379651/ipykernel_3264528/3705899607.py:6: DtypeWarning: Columns (2,4,6,8,12,14,15,21,30,31,32,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  binding_pockets_df = pd.read_csv(AF2BIND_PREDICTIONS_PATH, sep=",", header=0)


uniprot    15026
dtype: int64

In [3]:
import csv, pickle

DATA_DIRECTORY = f'{PATH}/data-extraction'
with open(f'{DATA_DIRECTORY}/LIGYSIS-files/LIGYSIS_sites_DEF_TRANS.pkl', 'rb') as file:
    mapping_df = pickle.load(file)
    mapping_df['rep_chain'] = mapping_df['pdb_id'] + '_' + mapping_df['auth_asym_id'] # or struct_asym_id???
    mapping_df = mapping_df[['rep_chain', 'ACC', 'up_aas', 'aas']]
with open(f'{DATA_DIRECTORY}/LIGYSIS-files/ALL_PREDS_COMBINED_RIGHT_EXTENDED_DEF.pkl', 'rb') as file:
    results_df = pickle.load(file)
    results_df = results_df[results_df['origin'] == 'LIGYSIS']
    results_df = results_df[['rep_chain']]

results_df = results_df.merge(mapping_df, on='rep_chain', how='left')
rep_chain_to_uniprot_mapping = dict(zip(results_df['rep_chain'], results_df['ACC']))
print("Number of structures in LIGYSIS:", len(rep_chain_to_uniprot_mapping))

print()
missing_in_af2bind = 0
list_of_predcited_up = binding_pockets_df['uniprot'].unique().tolist()
with open(f'{DATA_DIRECTORY}/ligysis_for_residue_level_evaluation.csv', 'r') as file:
    # with open(f'{DATA_DIRECTORY}/af2bind_predictions.csv', 'w') as output_file:
    reader = csv.reader(file, delimiter=';')
    for row in reader:
        rep_chain = row[0] + '_' + row[1]
        uniprot_id = rep_chain_to_uniprot_mapping[rep_chain]
        if uniprot_id not in list_of_predcited_up:
            missing_in_af2bind += 1
print("Number of structures in LIGYSIS missing in AF2Bind predictions:", missing_in_af2bind)

Number of structures in LIGYSIS: 2775

Number of structures in LIGYSIS missing in AF2Bind predictions: 498
